# Apply OSM-built bakery name classifier on FHRS dataset

In [ ]:
from pathlib import Path

import joblib
import pandas as pd


FHRS_DATE = "2026-07-23"

INTERIM_FOLDER = Path("../data/interim")
MODEL_FOLDER = Path("../models")

FHRS_PATH = (INTERIM_FOLDER/f"london_fhrs_prepared_{FHRS_DATE}.csv")

MODEL_PATH = (MODEL_FOLDER/f"osm_bakery_name_classifier.joblib")

fhrs_business_names = pd.read_csv(FHRS_PATH, usecols=["BusinessName", "BusinessNameClean"])

bakery_model = joblib.load(MODEL_PATH)

print(f"FHRS establishment rows: {len(fhrs_business_names)}")

print("Bakery name classifier loaded successfully.")

FHRS establishment rows: 81080
Bakery name classifier loaded successfully.


In [35]:
fhrs_business_names = (fhrs_business_names[
    fhrs_business_names["BusinessNameClean"].notna()
    & fhrs_business_names["BusinessNameClean"].ne("")
    ].copy()
)

fhrs_unique_names = (fhrs_business_names
    .groupby("BusinessNameClean", as_index=False)
    .agg(DisplayName=("BusinessName", "first"),
         StoreCount=("BusinessName", "size")))

print(f"FHRS rows: {len(fhrs_business_names)}")
print(f"Unique cleaned names: {len(fhrs_unique_names)}")

fhrs_unique_names.sample(10)

FHRS rows: 81080
Unique cleaned names: 65799


,BusinessNameClean,DisplayName,StoreCount
47787,rumble tumble ltd,Rumble Tumble Ltd,1
5853,belmont united synagogue,Belmont United Synagogue,1
33364,londis sutton court,Londis - Sutton Court,1
23632,h a mcparland chemists ltd,H A Mcparland (Chemists) Ltd,1
25479,hindu tamil cultural association,Hindu Tamil Cultural Association,1
26848,impact food bidco limited,IMPACT FOOD BIDCO LIMITED,1
39335,new lee ho fook,New Lee Ho Fook,1
19583,fete lounge,Fete Lounge,1
24660,hawd cafe hawd,Hawd Cafe / Hawd,1
37237,molten toffee,Molten Toffee,2


In [ ]:
fhrs_unique_names["BakeryProbability"] = (bakery_model.predict_proba(fhrs_unique_names["BusinessNameClean"])[:, 1])

fhrs_unique_names = (fhrs_unique_names.sort_values(["BakeryProbability", "StoreCount"], ascending=[False, False], ignore_index=True))

fhrs_unique_names.head(20)

,BusinessNameClean,DisplayName,StoreCount,BakeryProbability
0,bakery and cake,Bakery And Cake,1,0.999999
1,g o d cakes bakery,G.O.D Cakes / Bakery,1,0.999998
2,bakers cakes,Bakers Cakes,1,0.999997
3,cake house bakery,Cake House Bakery,5,0.999996
4,q s bakery,Q's Bakery,1,0.999995
5,b bakery,B.Bakery,1,0.999994
6,the bakery,The Bakery,2,0.999993
7,b j bakery,B.J Bakery,1,0.999992
8,east cake and bakery ltd,East Cake & Bakery Ltd,1,0.999991
9,q s bakery ltd,Q's Bakery Ltd,1,0.999989


In [37]:
candidate_results = []

for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    candidates = fhrs_unique_names[fhrs_unique_names["BakeryProbability"].ge(threshold)]

    candidate_results.append({
        "Threshold": threshold,
        "UniqueNamesToReview": len(candidates),
        "FHRSLocationsRepresented": candidates["StoreCount"].sum(),
    })

candidate_counts = pd.DataFrame(candidate_results)

candidate_counts

,Threshold,UniqueNamesToReview,FHRSLocationsRepresented
0,0.5,4957,5753
1,0.6,3936,4609
2,0.7,3241,3737
3,0.8,2657,3059
4,0.9,2112,2462


In [32]:
OUTPUT_PATH = (INTERIM_FOLDER/ f"london_fhrs_business_names_scored_{FHRS_DATE}.csv")

fhrs_unique_names.to_csv(OUTPUT_PATH,index=False,)

print(f"Scored FHRS names saved to: {OUTPUT_PATH}")

Scored FHRS names saved to: ..\data\interim\london_fhrs_business_names_scored_2026-07-23.csv


In [52]:
fhrs_unique_names[
    fhrs_unique_names["BakeryProbability"].between(0.85, 0.90, inclusive="left")
    ].sample(n=30)

,BusinessNameClean,DisplayName,StoreCount,BakeryProbability
2367,pareezcakes,Pareezcakes,1,0.855063
2250,break bread community lunch,Break Bread- community lunch,1,0.875939
2131,lola s cupcakes clapham junction,Lola's Cupcakes Clapham Junction,1,0.895541
2240,shabalicious bakes,Shabalicious Bakes,1,0.877107
2180,eggfree cake box worcester park,Eggfree Cake Box Worcester Park,1,0.887190
2209,baked by nikii,Baked by Nikii,1,0.883601
2375,lola s cupcakes broadgate,Lola's Cupcakes Broadgate,1,0.853999
2236,baked bliss,Baked Bliss,1,0.877576
2380,chikz patisserie limited,Chikz Patisserie Limited,1,0.852812
2150,anns preserves and bakes,Anns preserves and bakes,1,0.892197


In [53]:
fhrs_unique_names[
    fhrs_unique_names["BakeryProbability"].between(0.80, 0.85, inclusive="left")
    ].sample(n=30)

,BusinessNameClean,DisplayName,StoreCount,BakeryProbability
2649,grodz,Grodz,2,0.800847
2536,wapping sourdough,Wapping Sourdough,2,0.821886
2595,the cambridge theatre,The Cambridge Theatre,1,0.811694
2468,cakey kel,Cakey Kel,1,0.834473
2633,irene,Irene,2,0.804305
2516,onsu,Onsu,1,0.826699
2606,ramada london north,Ramada London North,1,0.809620
2431,pie crust cafe,Pie Crust Cafe,1,0.841248
2584,jermyn street theatre,Jermyn Street Theatre,1,0.813302
2564,jolene,Jolene,1,0.817253


In [55]:
fhrs_unique_names[
    fhrs_unique_names["BakeryProbability"].between(0.75, 0.80, inclusive="left")
    ].sample(n=30)

,BusinessNameClean,DisplayName,StoreCount,BakeryProbability
2904,crumb,CRUMB,1,0.758612
2911,london 6dana ltd,London 6Dana Ltd,1,0.756805
2737,that s cake by susanne,That’s Cake! by Susanne,1,0.788013
2751,erika s patties,Erika's Patties,1,0.785347
2742,oven or nothing,OVEN OR NOTHING,1,0.786901
2727,aamina s sweet treats,Aamina's Sweet Treats,1,0.789953
2818,edami,EDAMI,1,0.773105
2788,london school of theology,London School of Theology,1,0.778456
2905,the cake topper factory,The Cake Topper Factory,1,0.758348
2944,yviescupkakes,Yviescupkakes,1,0.750657


In [56]:
fhrs_unique_names[
    fhrs_unique_names["BakeryProbability"].between(0.75, 0.80, inclusive="left")
    ].sample(n=30)

,BusinessNameClean,DisplayName,StoreCount,BakeryProbability
2845,up the crust,Up the Crust,1,0.768477
2727,aamina s sweet treats,Aamina's Sweet Treats,1,0.789953
2699,rise rise london ltd,RISE ( Rise London Ltd),1,0.793840
2736,cakies kitchen,Cakies kitchen,1,0.788081
2881,littleville,Littleville,1,0.763204
2878,like home ltd,Like Home Ltd,1,0.763793
2810,humble crumble ltd,Humble crumble ltd,1,0.774519
2757,the lyceum theatre,The Lyceum Theatre,1,0.784764
2917,freads breads,Freads Breads,1,0.755419
2911,london 6dana ltd,London 6Dana Ltd,1,0.756805
